# Notebook 00: System Architecture - End-to-End Walkthrough

## Project: Multi-Model Hybrid Movie Recommendation System

### Purpose of This Notebook

This notebook is a **reference document** - no code is executed here. It explains the complete system architecture across all three retrieval models so you understand exactly what happens at every stage.

### Three Retrieval Models

| Model | Approach | User Representation | Key Strength |
|-------|----------|-------------------|--------------|
| **Two-Tower** | Static collaborative filtering | Single 128-dim embedding from user profile | Robust baseline, fastest inference |
| **ComiRec** | Multi-interest capsule network | 4 separate 128-dim interest embeddings | Captures diverse tastes, best diversity |
| **SASRec** | Transformer self-attention on sequences | Single 128-dim context-aware embedding | Captures temporal patterns and recency |

All three share the same downstream pipeline: FAISS retrieval followed by XGBoost LambdaMART re-ranking. They differ **only** in how the user query vector is produced.

### Sections

1. The Full Pipeline - training, embedding extraction, FAISS indexing
2. How Each Retrieval Model Works - architecture and training details
3. Concrete Example - User 42 through each model
4. Latency Budget - how fast each step is and why
5. Cold-Start Strategy - handling new users
6. FAISS Deep Dive - index internals and search mechanics
7. Notebook Dependency Graph - what each notebook produces and consumes
8. FAQ

## Section 1: The Full Pipeline

All three models follow the same two-stage architecture. The only difference is Step 2 (how user embeddings are produced).

```
+=============================================================================+
|                         OFFLINE PHASE (Training + Index Building)            |
+=============================================================================+
|                                                                             |
|  Step 1: Feature Engineering (Notebook 02)                                  |
|  -----------------------------------------                                  |
|  Raw Data --> User Features (24-dim per user)                               |
|          --> Item Features (73-dim per movie)                                |
|          --> Cross Features (7-dim per user-movie pair)                      |
|          --> Negative Samples (4 negatives per positive)                     |
|          --> Sequential Interaction Histories (for ComiRec + SASRec)         |
|                                                                             |
|  Step 2: Retrieval Model Training (Notebooks 03 / 06 / 09)                  |
|  -----------------------------------------------------------                |
|                                                                             |
|  MODEL A: Two-Tower (Notebook 03)                                           |
|    User Features --> [User Tower DNN] --> 128-dim user embedding             |
|    Item Features --> [Item Tower DNN] --> 128-dim item embedding             |
|    Loss = BCE(sigmoid(dot(user_emb, item_emb)), label)                      |
|    Output: Single embedding per user, single embedding per item             |
|                                                                             |
|  MODEL B: ComiRec (Notebook 06)                                             |
|    User Sequence --> [Multi-Interest Capsule Network] --> 4 x 128-dim       |
|    Item Features --> [Item Encoder] --> 128-dim item embedding               |
|    Loss = BCE on max(dot(interest_k, item_emb) for k=1..4)                  |
|    Output: 4 interest vectors per user, single embedding per item           |
|                                                                             |
|  MODEL C: SASRec (Notebook 09)                                              |
|    User Sequence --> [Transformer Encoder (2 layers, 2 heads)]              |
|                  --> 128-dim context-aware embedding (last position)         |
|    Item Features --> [Item Projection] --> 128-dim item embedding            |
|    Loss = BCE(sigmoid(dot(sequence_emb, next_item_emb)), label)             |
|    Output: Single sequence-aware embedding per user                         |
|                                                                             |
|  Step 3: Extract & Store Item Embeddings (end of each model notebook)       |
|  --------------------------------------------------------------------       |
|  For ALL 21,082 movies in catalog:                                          |
|    item_features[movie_id] --> [Trained Item Encoder] --> 128-dim embedding  |
|  Store in FAISS index for similarity search.                                |
|  Each model has its own FAISS index (same items, different embedding space). |
|                                                                             |
|  Step 4: Extract & Store User Embeddings                                    |
|  -----------------------------------------                                  |
|  For ALL 138,002 known users:                                               |
|    Two-Tower: user_features --> [User Tower] --> 1 x 128-dim                |
|    ComiRec:   user_sequence --> [Capsule Net] --> 4 x 128-dim               |
|    SASRec:    user_sequence --> [Transformer] --> 1 x 128-dim               |
|  Store in embedding KV (Redis/DynamoDB in prod, numpy arrays for us).       |
|                                                                             |
|  Step 5: XGBoost Ranker Training (Notebooks 04 / 07 / 10)                   |
|  ----------------------------------------------------------                 |
|  For each (user, candidate, label) in train set:                            |
|    features = retrieval_score + user_features(24) + item_features(73)       |
|             + cross_features(7)                                              |
|    Two-Tower/SASRec: 105 features (1 retrieval score)                       |
|    ComiRec: 109 features (1 max + 4 per-interest retrieval scores)          |
|  Train XGBoost LambdaMART (rank:ndcg objective).                            |
|  Output: xgboost_ranker.json per model                                      |
|                                                                             |
+=============================================================================+
```

### After offline phase completes, these stores are populated (per model):

| Store | Content | Serves | Technology (prod / us) |
|---|---|---|---|
| Vector index | 21K item embeddings (128-dim) | Candidate retrieval (FAISS search) | Milvus, Pinecone / FAISS in-memory |
| Embedding KV | 138K user embeddings | Query vector(s) for FAISS | Redis, DynamoDB / numpy arrays |
| Feature KV (users) | 138K user profiles (24-dim) | XGBoost ranking input | Redis, DynamoDB / parquet |
| Feature KV (items) | 21K movie profiles (73-dim) | XGBoost ranking input | Redis, DynamoDB / parquet |
| XGBoost model | Tree ensemble | Re-ranking | Model file / in-memory Booster |

```
+=============================================================================+
|                         ONLINE PHASE (Inference)                             |
+=============================================================================+
|                                                                             |
|  User visits platform --> trigger recommendation request                    |
|                                                                             |
|  Stage A: Model Selection (Notebook 12 - Production Service)                |
|  -----------------------------------------------------------                |
|  Route user to retrieval model based on profile:                            |
|    - Eclectic users (high genre entropy, >50 ratings) --> ComiRec           |
|    - Heavy sequential users (>200 ratings) --> SASRec                       |
|    - Default (light/focused users) --> Two-Tower                            |
|                                                                             |
|  Stage B: Candidate Generation                                              |
|  --------------------------------                                            |
|  TWO-TOWER / SASREC:                                                        |
|    1. Look up user's 128-dim embedding from KV                              |
|    2. FAISS.search(user_embedding, K=200) --> top-200 item positions        |
|                                                                             |
|  COMIREC:                                                                   |
|    1. Look up user's 4 x 128-dim interest embeddings from KV               |
|    2. For each interest k=1..4:                                             |
|         FAISS.search(interest_k, K=50) --> 50 items per interest            |
|    3. Union and deduplicate --> ~150-200 unique candidates                  |
|    (Multi-probe retrieval: explores 4 different neighborhoods)              |
|                                                                             |
|  Stage C: Ranking                                                           |
|  --------------------------------                                            |
|  3. Look up user_features (24-dim) from Feature KV                          |
|  4. Look up item_features (73-dim) for each candidate from Feature KV       |
|  5. Compute cross_features (7-dim) and retrieval_score on the fly           |
|  6. Concatenate into 105/109-dim feature vector per candidate               |
|  7. XGBoost.predict(candidates x features) --> scores                       |
|                                                                             |
|  Stage D: Post-Processing                                                   |
|  --------------------------------                                            |
|  8. Filter already-seen items                                               |
|  9. MMR diversity re-ranking (penalize similarity to already-selected)      |
|  10. Return top-10 to user with metadata                                    |
|                                                                             |
+=============================================================================+
```

### Why two stages?

The catalog has 21,000+ movies (millions in production). Running XGBoost on all items per request is too slow. FAISS narrows 21K to ~200 candidates in <1ms via vector similarity, then XGBoost scores only those candidates with full feature richness.

## Section 2: How Each Retrieval Model Works

### Model A: Two-Tower (Static Collaborative Filtering)

The simplest and most widely deployed retrieval model. Each entity (user, item) is encoded independently by its own "tower" (a feed-forward neural network). The dot product of their embeddings predicts interaction probability.

```
User Tower:                              Item Tower:
  Input: user_features (24-dim)            Input: item_features (73-dim)
  --> Linear(24, 256) + ReLU + BN          --> Linear(73, 256) + ReLU + BN
  --> Linear(256, 128) + ReLU + BN         --> Linear(256, 128) + ReLU + BN
  --> Linear(128, 128) + L2Normalize       --> Linear(128, 128) + L2Normalize
  Output: 128-dim unit vector              Output: 128-dim unit vector

Score = dot(user_emb, item_emb)
Loss = BCE(sigmoid(score), label)
```

**Key property**: Towers are independent. Item embeddings can be pre-computed once and indexed in FAISS. At inference, only the user embedding lookup + FAISS search is needed (no neural network runs online).

**What it captures**: Users with similar rating patterns get similar embeddings (collaborative signal). Content features (genres, genome tags) provide regularization and cold-start capability.

**Limitation**: One embedding per user means it cannot represent diverse interests. A user who likes both horror and romance gets a "compromise" vector somewhere between the two clusters.

---

### Model B: ComiRec (Multi-Interest Network with Capsule Routing)

ComiRec addresses the single-embedding limitation by producing **multiple interest vectors** per user. Each interest captures a different facet of the user's taste (e.g., one for sci-fi, one for comedy, one for documentaries).

```
User Sequence: [item_1, item_2, ..., item_T]  (last 50 interactions)
    |
    v
Item Embedding Lookup --> (T, 128) sequence matrix
    |
    v
Dynamic Routing (3 iterations):
    For each of K=4 interest capsules:
      - Initialize routing coefficients uniformly
      - Iteratively refine: items that "agree" with a capsule get routed to it
      - Each capsule output = weighted sum of item embeddings
    |
    v
4 x 128-dim interest embeddings (L2-normalized)

Score = max(dot(interest_k, item_emb) for k=1..4)
Loss = BCE(sigmoid(max_score), label)
```

**Key property**: Multi-probe retrieval. Each interest vector independently queries FAISS, retrieving items from 4 different neighborhoods. The union of results is more diverse than any single query could produce.

**What it captures**: A user who watches sci-fi AND cooking shows will have separate interest capsules for each. FAISS returns sci-fi candidates from one probe and cooking candidates from another.

**Trade-off**: 4x the FAISS queries per request (still fast: 4 x <1ms), but much higher diversity. Retrieval recall is lower per-probe but higher in aggregate across diverse candidate pools.

---

### Model C: SASRec (Self-Attentive Sequential Recommendation)

SASRec treats recommendation as a sequence prediction problem. It uses a Transformer encoder to model the user's recent interaction sequence, producing a single embedding that captures temporal context and item-to-item transitions.

```
User Sequence: [item_1, item_2, ..., item_T]  (last 50 interactions, chronological)
    |
    v
Item Embedding Lookup + Positional Encoding --> (T, 128)
    |
    v
Transformer Encoder (2 layers, 2 attention heads):
    Layer 1: MultiHeadAttention(Q=K=V=sequence) + FFN
    Layer 2: MultiHeadAttention(Q=K=V=hidden) + FFN
    Causal mask: each position can only attend to earlier positions
    |
    v
Take last position output --> 128-dim context-aware embedding (L2-normalized)

Score = dot(sequence_emb, next_item_emb)
Loss = BCE(sigmoid(score), label)  (predict next item at each position)
```

**Key property**: The embedding is **dynamic** -- it changes based on what the user recently watched. If the user just watched 3 horror movies in a row, the embedding shifts toward the horror region, even if their long-term preference is sci-fi.

**What it captures**: Sequential patterns ("users who watch A then B often watch C next"), session-level intent, and recency effects that static models miss.

**Trade-off**: Embeddings must be recomputed when new interactions arrive (cannot be cached indefinitely like Two-Tower). In practice, batch recomputation runs hourly/daily.

---

### Comparison Summary

| Aspect | Two-Tower | ComiRec | SASRec |
|--------|-----------|---------|--------|
| User embedding | 1 x 128-dim (static) | 4 x 128-dim (static) | 1 x 128-dim (dynamic) |
| Input signal | User profile features | Interaction sequence | Interaction sequence |
| FAISS queries/request | 1 | 4 (multi-probe) | 1 |
| Embedding staleness | OK (profile changes slowly) | OK (interests stable) | Stale faster (misses recent items) |
| Diversity mechanism | None (single point) | Multi-probe retrieval | None (single point) |
| Cold-start | Needs user features | Needs 3+ interactions | Needs 3+ interactions |
| Training data | Binary pairs (user, item, label) | Sequences of interactions | Sequences of interactions |
| Inference latency (P50) | 0.93ms | 1.48ms | 1.14ms |

## Section 3: Concrete Example - User 42 Through Each Model

User 42 is an established user with 125 ratings. They like Sci-Fi/Action, also watches some drama. Let's trace the request through each retrieval model.

---

### Path A: Two-Tower

**Step 1 - Look up pre-computed user embedding:**

```
embedding_kv[user_idx=42] = [0.15, -0.33, 0.72, ..., 0.28]  (128-dim)
```

This was computed offline by passing User 42's 24-dim features through the trained User Tower.

**Step 2 - FAISS search (single probe):**

```
FAISS.search(user_embedding, K=200) --> top-200 movies sorted by dot-product

Returns items from the Sci-Fi/Action neighborhood (User 42's "average" taste):
  Rank 1: "Blade Runner 2049" (Sci-Fi)       score=0.94
  Rank 2: "The Matrix Reloaded" (Sci-Fi)     score=0.91
  Rank 3: "Arrival" (Sci-Fi, Drama)           score=0.89
  ...
  Rank 200: "Gone Girl" (Thriller)            score=0.52
```

All 200 candidates come from one region of embedding space. High recall for Sci-Fi but misses User 42's occasional drama interest.

---

### Path B: ComiRec

**Step 1 - Look up 4 interest embeddings:**

```
embedding_kv[user_idx=42] = {
  interest_0: [0.22, -0.45, ...]  (Sci-Fi/Action capsule)
  interest_1: [0.18, 0.67, ...]   (Drama capsule)
  interest_2: [-0.11, 0.33, ...]  (Thriller capsule)
  interest_3: [0.05, -0.02, ...]  (low-activation, near-zero)
}
```

Capsule routing separated User 42's history into distinct taste clusters.

**Step 2 - FAISS search (4 probes, 50 each):**

```
Probe 0 (Sci-Fi): "Blade Runner 2049", "Ex Machina", "Interstellar", ...
Probe 1 (Drama):  "Schindler's List", "Shawshank Redemption", "Parasite", ...
Probe 2 (Thriller): "Se7en", "Gone Girl", "Zodiac", ...
Probe 3 (low activation): skipped (norm < 0.01)

Union: ~143 unique candidates from 3 diverse neighborhoods
```

The candidate set is inherently more diverse -- it includes items from multiple taste regions that a single Two-Tower query would miss.

---

### Path C: SASRec

**Step 1 - Look up sequence-aware embedding:**

```
embedding_kv[user_idx=42] = [0.31, -0.22, 0.55, ..., 0.19]  (128-dim)
```

This was computed by running User 42's last 50 interactions through the Transformer. If their recent watches were sci-fi heavy, this embedding is shifted toward sci-fi (more so than Two-Tower's static average).

**Step 2 - FAISS search (single probe):**

```
FAISS.search(user_embedding, K=200) --> top-200 movies

Returns items biased toward RECENT interests:
  Rank 1: "Dune: Part Two" (Sci-Fi)          score=0.91
  Rank 2: "Everything Everywhere..." (Sci-Fi) score=0.88
  Rank 3: "Oppenheimer" (Drama, recent)       score=0.86
  ...
```

The self-attention mechanism gives higher weight to recent items, so the embedding reflects current mood rather than lifetime average.

---

### After Retrieval: Same Ranking Pipeline for All Three

Regardless of which model retrieved candidates, the ranking stage is identical:

```
For each candidate (from whichever model):
  1. user_features[42]       --> 24-dim (same for all candidates)
  2. item_features[movie_id] --> 73-dim (per candidate)
  3. cross_features          --> 7-dim (genre_match, pop_gap, time features)
  4. retrieval_score         --> 1-dim (dot product from retrieval)
     (ComiRec: 5-dim = max + 4 per-interest scores)

  XGBoost input = concat(retrieval_score, user, item, cross) = 105 or 109 dim
  XGBoost.predict() --> final score

Return top-10 sorted by XGBoost score, after MMR diversity re-ranking.
```

## Section 3b: Why the Ranking Changes Everything

A critical insight from our experiments: the **retrieval model determines the candidate pool diversity**, but the **XGBoost ranker dominates final ranking quality**.

### Evidence from feature importance:

In all three XGBoost rankers, the retrieval_score (dot product from FAISS) ranks only #10-#15 in feature importance. The top features are always:
- `item_avg_rating_norm` (importance ~154) -- item quality dominates
- `genre_match` (importance ~80) -- user-item compatibility
- `genome_pca` dimensions -- content similarity from tag genome

This means: the retrieval model's job is to produce a **diverse, high-quality candidate pool**. It does NOT need to perfectly rank items -- XGBoost handles that with richer features.

### Why ComiRec wins end-to-end despite lower retrieval recall:

```
Two-Tower retrieval: 200 candidates, all from one region
  --> XGBoost picks best 10, but they're similar to each other

ComiRec retrieval: ~150 candidates from 4 diverse regions
  --> XGBoost picks best 10, and they span multiple taste facets
  --> Higher final NDCG because test positives are diverse too
```

This is the multi-model architecture's key design insight: separate the "what to consider" decision (retrieval) from the "how to rank" decision (XGBoost).

## Section 5: Cold-Start Strategy

New users have no pre-computed embedding. The system handles this through a progressive ramp that differs by model.

---

### Stage 1: True Cold Start (zero signals) -- All models

User just signed up. No browsing, no clicks.

```
--> Serve pre-computed trending/popular list (cached, refreshed hourly).

No model runs. Just serve a cached list.
Output: Forrest Gump, Pulp Fiction, Shawshank Redemption, ...
```

---

### Stage 2: Session-Based (2-5 browsed items)

User has clicked on a few movies but has not rated anything.

**Two-Tower approach:**
```
Look up item embeddings for browsed movies.
Average them to create a pseudo-user embedding.
FAISS.search(average_embedding, K=200) --> lookalike movies
```

**ComiRec approach:**
```
Cannot form meaningful capsules from 2-5 items (need diversity to route).
Fallback: treat each browsed item as a separate "interest" and probe FAISS individually.
```

**SASRec approach:**
```
Run the Transformer on the short sequence [item_1, ..., item_5].
Positional encoding still works. Self-attention still runs.
Output embedding is valid but noisy (short context).
```

---

### Stage 3: Established (10+ ratings) -- Models diverge

**Two-Tower**: Compute user_features from ratings, run User Tower once, store embedding. Full personalization.

**ComiRec**: Sequence is long enough for capsule routing to separate interests. All 4 capsules become active and meaningful.

**SASRec**: Sequence provides enough context for attention patterns to emerge. Temporal patterns become detectable.

---

### Summary

| Stage | Signals | Two-Tower | ComiRec | SASRec |
|---|---|---|---|---|
| 1. Cold start | None | Popular list | Popular list | Popular list |
| 2. Session | 2-5 items | Avg item embs | Per-item probes | Short-seq Transformer |
| 3. Established | 10+ ratings | Full personalization | 4 interest capsules | Full sequence model |
| 4. Heavy user | 50+ ratings | Same as stage 3 | Best diversity | Best recency capture |

**Key insight**: Two-Tower handles cold-start most gracefully (needs only user features, not sequences). ComiRec and SASRec need at minimum ~5 interactions before they outperform the simpler model.

## Section 6: FAISS Deep Dive

FAISS (Facebook AI Similarity Search) is the vector index that powers candidate generation for all three models. Each model has its own FAISS index (same items, different embedding spaces).

### What FAISS stores (per model):

```
IndexFlatIP - stores ONLY the embedding vectors:
+=======================================================================+
| Position |  128-dim Embedding (float32)                                |
+==========+=============================================================+
| 0        | [0.12, -0.45, 0.78, ..., 0.33]   (movie_idx 0)             |
| 1        | [-0.22, 0.67, 0.11, ..., -0.55]  (movie_idx 1)             |
| 2        | [0.89, 0.01, -0.34, ..., 0.22]   (movie_idx 2)             |
| ...      | ...                                                         |
| 21,081   | [0.44, -0.12, 0.56, ..., 0.91]   (movie_idx 21081)         |
+=======================================================================+

Total per index: 21,082 x 128 x 4 bytes = ~10.8 MB
Three indices total: ~32 MB
```

FAISS uses **positional indexing** -- it returns integer positions (0, 1, 2, ...), not movie IDs. We maintain a separate mapping:

```
position 5820 --> idx2movie[5820] --> movieId 72998 (original MovieLens ID)
```

### How search works differently for each model:

**Two-Tower / SASRec (single-probe):**
```
Query: user_embedding = [0.15, -0.33, ..., 0.28]  (128-dim, 1 vector)
FAISS computes: inner_product(query, every_stored_vector)
Returns: top-200 positions + scores
One search call, one result set.
```

**ComiRec (multi-probe):**
```
For each interest k = 0..3:
  Query: interest_k = [0.22, -0.45, ..., 0.11]  (128-dim)
  FAISS computes: inner_product(interest_k, every_stored_vector)
  Returns: top-50 positions + scores

Union all results, deduplicate --> ~150-200 unique candidates
Score each candidate by max(dot(interest_k, item_emb) for all k)
```

### Why Inner Product instead of L2 distance?

All embeddings are L2-normalized (unit vectors). For unit vectors:
- inner product = cosine similarity
- Scores fall in [-1, +1]
- Rankings are identical to cosine similarity
- IP is faster to compute than L2 (no square root)

### Index types at different scales:

| Index Type | Speed | Recall | Use case |
|---|---|---|---|
| IndexFlatIP | Exact | 100% | <100K items (our case: 21K) |
| IndexIVFFlat | ~10x faster | ~95-99% | 100K - 10M items |
| IndexHNSW | ~50x faster | ~95% | 10M+ items |

For 21K movies, brute-force exact search completes in <1ms. Production systems with millions of items use approximate indices that trade small recall loss for major speed gains.

## Section 4: Latency Budget

The entire recommendation must complete in under 50ms for a good user experience. Here is where time goes for each model:

```
TWO-TOWER PATH
================================================================
Step                          | Our Dataset    | Production (10M items)
------------------------------|----------------|------------------------
1. User embedding lookup      | <0.1 ms        | <1 ms (Redis)
2. FAISS search (K=200)       | <1 ms          | ~5-10 ms (IVF/HNSW)
3. User features lookup       | <0.1 ms        | <1 ms (Redis)
4. Item features lookup (x200)| <0.5 ms        | ~2 ms (Redis batch)
5. XGBoost predict (200 rows) | ~2 ms          | ~2 ms (tree traversal)
6. MMR post-processing        | ~0.5 ms        | ~0.5 ms
------------------------------|----------------|------------------------
TOTAL                         | ~3.3 ms        | ~12-18 ms


COMIREC PATH
================================================================
Step                          | Our Dataset    | Production (10M items)
------------------------------|----------------|------------------------
1. Interest embeddings lookup | <0.1 ms        | <1 ms (Redis, 4 vecs)
2. FAISS search x4 (K=50 ea) | <1.5 ms        | ~15-20 ms (4 probes)
3. Deduplication + union      | <0.1 ms        | <0.5 ms
4-6. Same as Two-Tower        | ~3 ms          | ~5 ms
------------------------------|----------------|------------------------
TOTAL                         | ~4.2 ms        | ~22-28 ms


SASREC PATH
================================================================
Step                          | Our Dataset    | Production (10M items)
------------------------------|----------------|------------------------
1. User embedding lookup      | <0.1 ms        | <1 ms (Redis)
2. FAISS search (K=200)       | <1 ms          | ~5-10 ms (IVF/HNSW)
3-6. Same as Two-Tower        | ~3 ms          | ~5 ms
------------------------------|----------------|------------------------
TOTAL                         | ~3.5 ms        | ~12-18 ms
```

### Measured latencies from production simulation (Notebook 12):

| Metric | Two-Tower | ComiRec | SASRec |
|--------|-----------|---------|--------|
| P50 | 0.93 ms | 1.48 ms | 1.14 ms |
| P95 | 4.47 ms | 4.47 ms | 4.47 ms |
| Throughput | 305 req/s | 305 req/s | 305 req/s |

### Why this is fast:

- **No neural network at inference.** All embeddings are pre-computed offline. The online path is: KV lookup + FAISS search + KV batch lookup + XGBoost predict.

- **FAISS on 21K items is brute-force exact search in <1ms.** At production scale (10M+), you switch to approximate indices (IVF, HNSW) that trade ~2-5% recall for 10-50x speed.

- **XGBoost is fast for prediction.** Scoring 200 candidates through a tree ensemble is just if-else comparisons. No matrix multiplications, no GPU needed.

- **ComiRec is only ~50% slower despite 4 FAISS probes** because each probe is smaller (K=50 vs K=200) and FAISS search cost is sublinear.

## Section 7: Notebook Dependency Graph

```
Notebook 00 (this file) - Architecture Overview
  Produces: Nothing (reference document)
  Consumes: Nothing

Notebook 01 (EDA)
  Produces: Insights and design decisions
  Consumes: Raw data (ml-25m/*.csv)

Notebook 02 (Feature Engineering)
  Produces:
    data/processed/item_features.parquet      (21,082 x 73)
    data/processed/user_features.parquet      (138,002 x 24)
    data/processed/two_tower_train.parquet    (~71M pairs, binary labels)
    data/processed/train_set.parquet          (train interactions, original ratings)
    data/processed/val_set.parquet            (validation interactions)
    data/processed/test_set.parquet           (test interactions)
    data/processed/train_interaction_features.parquet
    data/processed/val_interaction_features.parquet
    data/processed/test_interaction_features.parquet
    data/processed/metadata.pkl               (ID mappings, dimensions)
    data/processed/genome_pca_model.pkl       (PCA + Scaler)
  Consumes: Raw data (ml-25m/*.csv)

---------- TWO-TOWER PIPELINE ----------

Notebook 03 (Two-Tower Model Training)
  Produces:
    models/two_tower_model.pt         (trained PyTorch model)
    models/faiss_index_128dim.bin     (FAISS index, 21K x 128-dim)
    models/item_embeddings_128dim.npy (21K x 128, item embeddings)
    models/user_embeddings_128dim.npy (138K x 128, user embeddings)
  Consumes: two_tower_train.parquet, item/user_features.parquet, metadata.pkl

Notebook 04 (Two-Tower XGBoost Ranking)
  Produces:
    models/xgboost_ranker.json        (LambdaMART ranker, 105 features)
    models/ranker_feature_names.pkl
  Consumes: train/val/test sets, features, NB03 embeddings

Notebook 05 (Two-Tower Evaluation)
  Produces: Metric reports and visualizations
  Consumes: NB03 + NB04 models, test set

---------- COMIREC PIPELINE ----------

Notebook 06 (ComiRec Model Training)
  Produces:
    models/comirec/comirec_model.pt       (capsule network)
    models/comirec/faiss_index.bin        (FAISS index, 21K x 128-dim)
    models/comirec/item_embeddings.npy    (21K x 128)
    models/comirec/user_embeddings.npy    (138K x 4 x 128, multi-interest)
  Consumes: Sequential interaction data, item/user features, metadata.pkl

Notebook 07 (ComiRec XGBoost Ranking)
  Produces:
    models/comirec/xgboost_ranker.json    (LambdaMART ranker, 109 features)
    models/comirec/ranker_feature_names.pkl
  Consumes: train/val/test sets, features, NB06 embeddings

Notebook 08 (ComiRec Evaluation)
  Produces: Metric reports, two-way comparison (TT vs ComiRec)
  Consumes: NB03/04 + NB06/07 models, test set

---------- SASREC PIPELINE ----------

Notebook 09 (SASRec Model Training)
  Produces:
    models/sasrec/sasrec_model.pt         (Transformer encoder)
    models/sasrec/faiss_index.bin         (FAISS index, 21K x 128-dim)
    models/sasrec/item_embeddings.npy     (21K x 128)
    models/sasrec/user_embeddings.npy     (138K x 128, sequence-aware)
  Consumes: Sequential interaction data, item/user features, metadata.pkl

Notebook 10 (SASRec XGBoost Ranking)
  Produces:
    models/sasrec/xgboost_ranker.json     (LambdaMART ranker, 105 features)
    models/sasrec/ranker_feature_names.pkl
  Consumes: train/val/test sets, features, NB09 embeddings

Notebook 11 (Three-Way Evaluation)
  Produces: Full three-way comparison (TT vs ComiRec vs SASRec)
  Consumes: All 3 retrieval + ranking models, test set

---------- PRODUCTION ----------

Notebook 12 (Production Inference Simulation)
  Produces: Latency benchmarks, monitoring report, request logs
  Consumes: All models + features (simulates full service)

Notebook 13 (A/B Testing Framework)
  Produces: Experiment results, statistical analysis, decision report
  Consumes: All models + test set ground truth
```

### Data flow diagram:

```
Raw Data (ml-25m)
    |
    v
NB02 (Feature Engineering)
    |
    +--> NB03 (Two-Tower) --> NB04 (XGBoost) --> NB05 (Eval)
    |                                               |
    +--> NB06 (ComiRec)  --> NB07 (XGBoost) --> NB08 (Eval)
    |                                               |
    +--> NB09 (SASRec)   --> NB10 (XGBoost) --> NB11 (3-way Eval)
    |                                               |
    +-----------------------------------------------+
    |                                               |
    v                                               v
NB12 (Production Service)                    NB13 (A/B Testing)
```

## Section 8: FAQ

**Q: Why three different retrieval models instead of just using the best one?**

Different models excel for different user segments. Two-Tower is best for light/focused users, ComiRec for eclectic users with diverse tastes, and SASRec for sequential patterns. The production service (NB12) routes users to the appropriate model based on their profile.

**Q: Do all three models produce the same embedding dimensionality?**

Yes, all use 128-dim embeddings. This is intentional -- it means all three share the same FAISS index type (IndexFlatIP with 128 dimensions), same XGBoost feature vector structure, and same downstream pipeline. The only difference is how the 128-dim vector is computed.

**Q: Why store BOTH 128-dim embeddings AND 24-dim/73-dim raw features?**

They serve different models. The 128-dim embedding is a compressed representation for FAISS similarity search (candidate generation). The raw features are what XGBoost needs for ranking -- tree splits operate on individual interpretable values (genre prefs, popularity, etc.), not opaque embedding dimensions.

**Q: Why can't Two-Tower use cross features?**

Because each tower encodes its entity independently. If the Item Tower needed to know which user is querying, we could not pre-compute item embeddings offline -- we would need to run it for every (user, item) pair per request (21K forward passes instead of 1 FAISS lookup). ComiRec and SASRec have the same constraint: the item embedding is independent of the query user.

**Q: How does ComiRec's multi-probe retrieval differ from just doing K=800 with Two-Tower?**

Fetching K=800 from a single embedding returns 800 items from one neighborhood (e.g., all sci-fi). ComiRec's 4 probes of K=50 each return items from 4 different neighborhoods (sci-fi, drama, thriller, comedy). The union has higher diversity even before any post-processing.

**Q: Why does SASRec produce a single embedding instead of one per sequence position?**

At inference, we only need the final position's output -- it summarizes the entire sequence context through self-attention. Using all positions would mean K x FAISS queries (expensive) for marginal benefit, since the final position already attends to all earlier positions via the causal mask.

**Q: Do we recommend movies the user already rated?**

No. A post-retrieval filter removes already-seen movies from the candidate set. We over-fetch from FAISS (K=200) to compensate for filtered items.

**Q: What if a new movie is added?**

Compute its 73-dim features, run it through each trained Item Encoder to get three 128-dim embeddings (one per model), add to each FAISS index. Immediately retrievable with no model retraining.

**Q: What if a user's behavior changes?**

- **Two-Tower**: Re-extract user features and run through User Tower (batch job, e.g., daily).
- **ComiRec**: Re-run capsule routing on updated sequence (batch job).
- **SASRec**: Most affected -- the sequence embedding encodes recency, so stale embeddings miss recent behavior. Needs more frequent updates (hourly in production).

**Q: Why XGBoost instead of a neural ranker (e.g., DCN, DeepFM)?**

XGBoost LambdaMART achieves NDCG@10 ~0.87 on this dataset with 105-109 features. Neural rankers need more data and training time to match this, and their marginal gain on 200 candidates (where the retrieval model already filtered to relevant items) is small. XGBoost also has interpretable feature importance, faster inference (no GPU), and simpler deployment.

**Q: What were the final results?**

| Model | Retrieval Recall@200 | End-to-End NDCG@10 | ILD (diversity) |
|-------|---------------------|-------------------|-----------------|
| Two-Tower | 0.305 (best) | 0.032 | 0.28 |
| ComiRec | 0.214 | 0.036 (best) | 0.54 (best) |
| SASRec | 0.264 | 0.033 | 0.41 |

ComiRec wins end-to-end despite lower retrieval recall because its diverse candidate pool gives XGBoost better material to work with.

---

*This notebook is a reference document. No code executes here. Proceed to Notebook 01 for EDA.*